In [ ]:
#| export
# Lazy type-hint evaluation so `Path` doesn't need to be imported
# just to put it in a function signature.
from __future__ import annotations

In [ ]:
#| default_exp web_app


# Step 9: Gallery endpoint + canonical `create_app`

![Step 9 diagram](https://raw.githubusercontent.com/jaewilson07/bird-watcher/main/docs/diagrams/09-step.png)

**Goal:** add a `/gallery` endpoint that returns the recent sightings as JSON, and mount the snapshot folder so the browser can show thumbnails.

We define the full factory `create_app` here — including `/`, `/health`, `/gallery`, and the static `/snapshots` mount. `bird_watcher/web_app.py` is now complete.

## Step 9.0 — Setup

In [ ]:
from pathlib import Path

import yaml

ROOT_CANDIDATES = [Path.cwd(), Path.cwd().parent]
PROJECT_ROOT = next(
    (root for root in ROOT_CANDIDATES if (root / "tutorials").exists()),
    Path.cwd(),
)
CONFIG_FILE = PROJECT_ROOT / "config.yaml"

CONFIG = {}
if CONFIG_FILE.exists():
    CONFIG = yaml.safe_load(CONFIG_FILE.read_text()) or {}
    if not isinstance(CONFIG, dict):
        raise TypeError(f"{CONFIG_FILE} must contain a top-level mapping")
else:
    print(f"Config file not found at {CONFIG_FILE}; using defaults.")

# Folder + file constants. _FOLDER = a directory, _FILE = a single file.
TUTORIALS_FOLDER = PROJECT_ROOT / "tutorials"
DATA_FOLDER = PROJECT_ROOT / "data"
SNAPSHOT_FOLDER = DATA_FOLDER / "snapshots"
CROP_FOLDER = DATA_FOLDER / "crops"
DB_FILE = DATA_FOLDER / "birds.db"
SAMPLE_BIRD_FILE = DATA_FOLDER / "samples" / "sample-bird.jpg"
MODEL_FILE = PROJECT_ROOT / "yolov8n.pt"

# From config.yaml (or hardcoded fallback)
PHONE_IP = str(CONFIG.get("phone_ip", "192.168.1.42"))
PHONE_URL = f"http://{PHONE_IP}:8080/photo.jpg"
SLACK_WEBHOOK = str(CONFIG.get("slack_webhook", ""))
HUGGINGFACE_API_KEY = str(CONFIG.get("huggingface_api_key", ""))

SNAPSHOT_FOLDER.mkdir(parents=True, exist_ok=True)
CROP_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Snapshot folder: {SNAPSHOT_FOLDER}")
print(f"Config file: {CONFIG_FILE}")
print(f"Phone URL: {PHONE_URL}")


## Step 9.1 — The factory + all four endpoints

In [ ]:
#| export
from pathlib import Path
from fastapi import FastAPI
from fastapi.staticfiles import StaticFiles
from bird_watcher.save_sighting import list_sightings_from_db

def create_app(
    db_file: Path,
    snapshot_folder: Path,
) -> FastAPI:
    """Build the full Bird Watcher FastAPI app.

    Endpoints:
        GET /          — landing page metadata
        GET /health    — uptime check
        GET /gallery   — recent sightings as JSON
        GET /snapshots/... — static thumbnails (mounted if folder exists)

    Args:
        db_file: path to the SQLite file with sightings.
        snapshot_folder: folder of saved JPEG snapshots (mounted for thumbnails).

    Returns:
        A configured FastAPI app, ready to be served with uvicorn.
    """
    from fastapi import FastAPI
    from fastapi.staticfiles import StaticFiles

    from bird_watcher.save_sighting import list_sightings_from_db

    app = FastAPI(title="Bird Watcher", version="0.0.0")

    @app.get("/")
    def index() -> dict:
        """Landing page."""
        return {"app": "bird-watcher", "status": "watching"}

    @app.get("/health")
    def health() -> dict:
        """Health check — useful for uptime monitoring."""
        return {"status": "ok"}

    @app.get("/gallery")
    def gallery(limit: int = 50) -> dict:
        """Recent sightings with thumbnails."""
        rows = list_sightings_from_db(db_file, limit=limit, verbose=False)
        return {"count": len(rows), "sightings": rows}

    if snapshot_folder.exists():
        app.mount(
            "/snapshots",
            StaticFiles(directory=str(snapshot_folder)),
            name="snapshots",
        )

    return app


## Step 9.2 — Smoke-test the app

In [ ]:
from fastapi.testclient import TestClient
from bird_watcher.web_app import create_app

app = create_app(DB_FILE, SNAPSHOT_FOLDER)
client = TestClient(app)

r = client.get("/")
assert r.status_code == 200
assert r.json()["app"] == "bird-watcher"

r = client.get("/health")
assert r.status_code == 200

r = client.get("/gallery")
assert r.status_code == 200
data = r.json()
assert "count" in data and "sightings" in data
print(f"✅ /, /health, /gallery all 200. /gallery returned {data['count']} sightings.")

## Acceptance criterion

All three endpoints respond with 200.

## What's next

**Step 10:** open [10-digest.ipynb](10-digest.ipynb) — we'll post a daily summary to Slack.